# 03-1. 오분류 이미지 보기 전용 노트북

이 노트북은 Excel 없이도 오분류 CSV를 보기 쉽게 확인하기 위한 파일입니다.

목표는 단순히 표를 보는 것이 아니라, **틀린 이미지와 true label, predicted label, confidence, breed를 함께 보면서 원인을 찾는 것**입니다.

## Step 0. 보는 기준을 정리합니다

오분류 분석에서는 다음 순서로 보는 것이 좋습니다.

1. confidence가 높은 오분류부터 봅니다.
2. true label과 predicted label이 어떤 방향으로 틀렸는지 봅니다.
3. breed를 같이 보면서 특정 품종에서 반복되는지 확인합니다.
4. 이미지를 직접 보고 실패 원인을 메모합니다.

confidence가 높은 오분류는 모델이 강하게 착각한 이미지이므로 특히 분석 가치가 있습니다.

## Step 1. 필요한 패키지와 경로를 준비합니다

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

MISCLASSIFIED_DIR = PROJECT_DIR / "outputs" / "misclassified"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ORIGINAL_CSV_PATH = MISCLASSIFIED_DIR / "regularized_original_misclassified.csv"
OPENCV_CSV_PATH = MISCLASSIFIED_DIR / "regularized_opencv_misclassified.csv"

ORIGINAL_CSV_PATH.exists(), OPENCV_CSV_PATH.exists()

## Step 2. 두 모델의 오분류 CSV를 불러옵니다

03에서 저장한 CSV를 다시 읽습니다. 두 CSV를 하나로 합쳐서 모델별로 비교할 수 있게 합니다.

In [ ]:
original_df = pd.read_csv(ORIGINAL_CSV_PATH)
opencv_df = pd.read_csv(OPENCV_CSV_PATH)

original_df["viewer_model"] = "Regularized Original"
opencv_df["viewer_model"] = "Regularized OpenCV"

all_misclassified_df = pd.concat([original_df, opencv_df], ignore_index=True)

summary_df = all_misclassified_df.groupby(["viewer_model", "true_label", "predicted_label"]).size().reset_index(name="count")
summary_df

## Step 3. 보고 싶은 조건을 선택합니다

아래 값을 바꾸면 원하는 오분류만 골라서 볼 수 있습니다.

- `MODEL_TO_VIEW`: `Regularized Original`, `Regularized OpenCV`, 또는 `all`
- `TRUE_LABEL_FILTER`: 실제 정답 기준 필터입니다. `cat`, `dog`, 또는 `None`
- `PRED_LABEL_FILTER`: 모델 예측 기준 필터입니다. `cat`, `dog`, 또는 `None`
- `BREED_TO_VIEW`: 특정 품종만 정확히 보고 싶을 때 사용합니다. 전체를 보려면 `None`으로 둡니다.
- `BREED_CONTAINS`: 품종 이름에 포함된 글자로 필터링합니다. 전체를 보려면 빈 문자열 `""`로 둡니다.
- `START_INDEX`: 다음 페이지를 볼 때 숫자를 12, 24처럼 바꿉니다.
- `MAX_IMAGES`: 한 번에 볼 이미지 개수입니다. `None`으로 두면 선택된 오분류 전체를 봅니다.

In [ ]:
MODEL_TO_VIEW = "Regularized Original"  # "Regularized Original", "Regularized OpenCV", "all"
TRUE_LABEL_FILTER = None                 # "cat", "dog", None
PRED_LABEL_FILTER = None                 # "cat", "dog", None
BREED_TO_VIEW = None                     # 예: "British Shorthair", "Sphynx", None
BREED_CONTAINS = ""                      # 예: "Terrier", "Siamese", ""
START_INDEX = 0
MAX_IMAGES = None  # None이면 선택된 오분류 전체를 저장/표시합니다.

safe_model_name = MODEL_TO_VIEW.lower().replace(" ", "_") if MODEL_TO_VIEW != "all" else "all_models"
if MODEL_TO_VIEW == "Regularized Original" and BREED_TO_VIEW is None:
    SAVE_GRID_PATH = FIGURE_DIR / "regularized_original_misclassified_grid.png"
else:
    safe_breed_name = "all_breeds" if BREED_TO_VIEW is None else BREED_TO_VIEW.lower().replace(" ", "_")
    SAVE_GRID_PATH = FIGURE_DIR / f"{safe_model_name}_{safe_breed_name}_misclassified_grid.png"

viewer_df = all_misclassified_df.copy()

if MODEL_TO_VIEW != "all":
    viewer_df = viewer_df[viewer_df["viewer_model"] == MODEL_TO_VIEW]

if TRUE_LABEL_FILTER is not None:
    viewer_df = viewer_df[viewer_df["true_label"] == TRUE_LABEL_FILTER]

if PRED_LABEL_FILTER is not None:
    viewer_df = viewer_df[viewer_df["predicted_label"] == PRED_LABEL_FILTER]

if BREED_TO_VIEW is not None:
    viewer_df = viewer_df[viewer_df["breed"] == BREED_TO_VIEW]

if BREED_CONTAINS:
    viewer_df = viewer_df[viewer_df["breed"].str.contains(BREED_CONTAINS, case=False, na=False)]

# confidence가 높은 오분류부터 봅니다.
viewer_df = viewer_df.sort_values("predicted_confidence", ascending=False).reset_index(drop=True)

print("선택된 오분류 개수:", len(viewer_df))
viewer_df[["viewer_model", "true_label", "predicted_label", "predicted_confidence", "breed", "image_path"]].head(20)

## Step 4. 이미지를 직접 확인합니다

제목에는 다음 정보가 표시됩니다.

- `true`: 실제 정답
- `pred`: 모델 예측
- `conf`: 모델이 자기 예측을 얼마나 확신했는지
- `breed`: 품종 이름

confidence가 높게 틀린 이미지는 모델이 잘못된 특징을 강하게 믿었을 가능성이 있습니다.

In [ ]:
def resolve_image_path(image_path):
    path = Path(image_path)
    if path.is_absolute():
        return path
    return PROJECT_DIR / path


def show_misclassified_grid(dataframe, start_index=0, max_images=12, cols=4, save_path=None):
    if max_images is None:
        page_df = dataframe.iloc[start_index:]
    else:
        page_df = dataframe.iloc[start_index:start_index + max_images]

    if page_df.empty:
        print("표시할 이미지가 없습니다. 필터 조건이나 START_INDEX를 확인합니다.")
        return

    rows = int(np.ceil(len(page_df) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, page_df.iterrows()):
        image_path = resolve_image_path(row["image_path"])
        image = Image.open(image_path).convert("RGB")

        ax.imshow(image)
        ax.axis("off")
        ax.set_title(
            f"{row['viewer_model']}\n"
            f"true: {row['true_label']} / pred: {row['predicted_label']}\n"
            f"conf: {row['predicted_confidence']:.2f}\n"
            f"breed: {row['breed']}",
            fontsize=9,
        )

    for ax in axes[len(page_df):]:
        ax.axis("off")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print("오분류 이미지 모음 저장:", save_path)

    plt.show()


show_misclassified_grid(viewer_df, start_index=START_INDEX, max_images=MAX_IMAGES, cols=4, save_path=SAVE_GRID_PATH)

## Step 5. 품종별 오분류 개수를 확인합니다

특정 품종에서 오분류가 많이 반복된다면, 모델이 그 품종의 특징을 dog/cat 구분에 충분히 활용하지 못했을 가능성이 있습니다.

아래 표에서 보고 싶은 품종 이름을 복사한 뒤, Step 3의 `BREED_TO_VIEW`에 넣고 Step 3~4를 다시 실행하면 해당 품종의 오분류 이미지만 볼 수 있습니다.

예시:

```python
BREED_TO_VIEW = "British Shorthair"
```

In [ ]:
breed_error_summary = (
    viewer_df.groupby(["true_label", "predicted_label", "breed"])
    .size()
    .reset_index(name="misclassified_count")
    .sort_values("misclassified_count", ascending=False)
)

breed_error_summary.head(20)

## Step 6. 원인 분석 메모용 표를 만듭니다

아래 표를 보고 직접 원인을 적으면 발표 자료에 바로 사용할 수 있습니다.

예시 원인:

- 얼굴 일부만 보임
- 배경이 복잡함
- 어두움 또는 흐림
- 품종 특성이 애매함
- 동물이 작게 찍힘
- 모델이 배경/털색에 과하게 반응한 것으로 추정

In [ ]:
analysis_memo_df = viewer_df[[
    "viewer_model",
    "true_label",
    "predicted_label",
    "predicted_confidence",
    "breed",
    "image_path",
]].head(30).copy()

analysis_memo_df["suspected_reason"] = ""
analysis_memo_df["improvement_idea"] = ""

memo_path = MISCLASSIFIED_DIR / "misclassified_analysis_memo_template.csv"
analysis_memo_df.to_csv(memo_path, index=False, encoding="utf-8-sig")

print("메모용 CSV 저장:", memo_path)
analysis_memo_df

## Step 7. 발표용 정리 문장

> 오분류 이미지는 confidence가 높은 순서로 확인했습니다. confidence가 높은 오분류는 모델이 단순히 헷갈린 것이 아니라 잘못된 시각적 특징을 강하게 믿었을 가능성이 있기 때문입니다. 이미지를 직접 확인하면서 얼굴 일부만 보이는 경우, 배경이 복잡한 경우, 품종 특성이 애매한 경우, 어둡거나 흐린 경우처럼 반복되는 실패 유형을 분류했습니다. 이를 바탕으로 head ROI/segmentation 활용, 실패 유형에 맞춘 증강, 전이학습 적용을 향후 개선 방향으로 정리했습니다.